# How to Align Multimodal Data (Beethoven)

**Figure 3 acid test** for TimeToAlign! — 16+ timelines across all 3 domains
(Physical, Logical, Graphical) in 6 `TimelineGroups`, 5 of them within one
`AlignmentBundle`.

**Structure:**
1. **Part I**: Build 3 recording groups (Groups 1-3) — 15 DPTs
2. **Part II**: Build Score group (Group 4) + align with recordings
3. **Part III**: Build Emerson group (Group 5) + cross-group coordinate transfer

## 0. Gold Standard Reference Values

| ID | Description | Samples | Rate | Grp |
|----|-------------|---------|------|-----|
| DPT1-5 | Normal | 11,753,638 / 11,195 / 22,389 / 45,844 / 63,965 | 44.1k / 42 / 84 / 172 / 240 | 1 |
| DPT6-10 | Mechanical | 12,426,696 / 11,836 / 23,671 / 48,469 / 67,628 | same rates | 2 |
| DPT11-15 | Exaggerated | 8,197,748 / 7,808 / 15,616 / 31,975 / 44,614 | same rates | 3 |

| Recording | Notes | Matched | Unmatched EEP | Unmatched ABC |
|-----------|-------|---------|---------------|---------------|
| Normal | 4,026 | 3,741 | 15 | 10 |
| Mechanical | 4,026 | 3,742 | 14 | 9 |
| Exaggerated | 2,820 | 2,650 | 4 | 1,101 |

## 1. Setup

In [1]:

import pandas as pd
from PIL import Image

from timetoalign import (
    ContinuousLogicalTimeline,
    ContinuousPhysicalTimeline,
    DiscreteGraphicalTimeline,
    Ms3Loader,
    NumberType,
    Provenance,
    RepoVizzLoader,
    TableMap,
    TimelineGroup,
    TimeUnit,
)
from timetoalign.alignment import AlignmentBundle, MatchLine, WarpMap
from timetoalign.alignment.matching import (
    match_notes_by_attributes,
    prepare_abc_notes_for_matching,
    prepare_eep_notes_for_matching,
)
from timetoalign.testdata import ensure_data
from timetoalign.timelines.types import SegmentLine

DATA_DIR = ensure_data("score") / "beethoven_op18-4iv_multimodal"

# XML manifest paths — the loader reads metadata from these files
NORMAL_XML = DATA_DIR / "StringQuartetEEP_I_Normal" / "StringQuartetEEP_I_Normal.xml"
MECHANICAL_XML = (
    DATA_DIR / "StringQuartetEEP_I_Mechanical" / "StringQuartetEEP_I_Mechanical.xml"
)
EXAGGERATED_XML = (
    DATA_DIR / "StringQuartetEEP_I_Exaggerated" / "StringQuartetEEP_I_Exaggerated.xml"
)

# Audio sources and instruments
AUDIO_SOURCES = [
    "mono",
    "binaural",
    "pickup_vln1",
    "pickup_vln2",
    "pickup_vla",
    "pickup_cello",
]
INSTRUMENTS = ["vln1", "vln2", "vla", "cello"]

Each EEP recording directory contains 5 modalities (audio, 3 feature types,
MoCap) plus `.notes` files with annotated note events. The function below
builds a `TimelineGroup` from one such directory via the XML manifest.

**Structure (per manuscript):**
- 5 parent physical timelines, each with a `SamplesToSeconds` c-map
- Audio, Tonal, LowLevel, Rhythm parents: 6 children each (mono, binaural, 4 pickups)
- MoCap parent: 4 children (one per instrument: vln1, vln2, vla, cello)

In [2]:
def build_recording_group(xml_path, group_id, group_name, dpt_base):
    """Build a TimelineGroup from one EEP recording directory via XML manifest.

    Args:
        xml_path: Path to the recording's XML manifest file.
        group_id: ID for the TimelineGroup.
        group_name: Human-readable name for the group.
        dpt_base: Starting DPT number (e.g. 1 for dpt_1-dpt_5).

    Returns:
        TimelineGroup with 5 hierarchical DPTs (parent + children).
    """
    rv = RepoVizzLoader.from_file(xml_path)
    n = dpt_base

    # 1. Audio (mono as parent, 6 sources as children). A child's id is led by
    # its parent's id: the three recordings share one bundle, and a timeline
    # id names one timeline there. The source label stays its name.
    audio = rv.create_timeline("mono", uid=f"dpt_{n}", name="Audio")
    for src in AUDIO_SOURCES:
        audio.add_child(
            rv.create_timeline(src, uid=f"{audio.id}:{src}", name=src), offset=0
        )

    # 2-4. Essentia descriptors (tonal, lowlevel, rhythm)
    desc_cfgs = [
        ("tonal", "ChordsStrength", 1),
        ("lowlevel", "Dissonance", 2),
        ("rhythm", "BeatsLoudness", 3),
    ]
    descriptors = []
    for desc_type, desc_name, offset in desc_cfgs:
        parent = rv.create_timeline(
            f"{desc_type}.{desc_name}.mono",
            uid=f"dpt_{n + offset}",
            name=desc_type.title(),
        )
        for src in AUDIO_SOURCES:
            parent.add_child(
                rv.create_timeline(
                    f"{desc_type}.{desc_name}.{src}",
                    uid=f"{parent.id}:{src}_{desc_type}",
                    name=f"{src}_{desc_type}",
                ),
                offset=0,
            )
        descriptors.append(parent)

    # 5. MoCap bb_angle (from the DescriptorGroup section of the XML)
    mocap = rv.create_timeline(
        rv.find_descriptor("bb_angle", "vln1"),
        uid=f"dpt_{n + 4}",
        name="MoCap",
    )
    for inst in INSTRUMENTS:
        child = rv.create_timeline(
            rv.find_descriptor("bb_angle", inst),
            uid=f"{mocap.id}:{inst}_mocap",
            name=f"{inst}_mocap",
        )
        mocap.add_child(child, offset=0)

    # Add notes to pickup children
    for inst in INSTRUMENTS:
        notes = rv.store.notes_for_instrument(inst)
        if notes and (pickup := audio.get_child(f"{audio.id}:pickup_{inst}")):
            pickup.add_events(notes.to_dataframe().to_dict("records"))

    return TimelineGroup(
        id=group_id,
        name=group_name,
        timelines=[audio, *descriptors, mocap],
    )

***
# Part I: Three Recording Groups (Groups 1-3)

Each EEP recording = 5 DPTs (audio + 3 feature types + MoCap) at different
sampling rates, all sharing the same physical duration. Note events live
as a child of the audio DPT.

## 2. Group 1: Normal Recording (DPT1-DPT5)

In [3]:
normal_group = build_recording_group(
    NORMAL_XML, "normal", "Normal Recording", dpt_base=1
)
normal_group

TimelineGroup(id='normal', n_timelines=5)

The audio timeline now carries the note annotations as a child:

In [4]:
normal_group.get_timeline("dpt_1")

DiscretePhysicalTimeline(id='dpt_1', length=11753638, unit=samples, events=0, children=6, cmaps=1)

## 3. Group 2: Mechanical Recording (DPT6-DPT10)

In [5]:
mechanical_group = build_recording_group(
    MECHANICAL_XML, "mechanical", "Mechanical Recording", dpt_base=6
)
mechanical_group

TimelineGroup(id='mechanical', n_timelines=5)

## 4. Group 3: Exaggerated Recording (DPT11-DPT15)

Shorter recording (~186s) — stops after measure 131.

In [6]:
exaggerated_group = build_recording_group(
    EXAGGERATED_XML,
    "exaggerated",
    "Exaggerated Recording",
    dpt_base=11,
)
exaggerated_group

TimelineGroup(id='exaggerated', n_timelines=5)

## 5. Part I Summary

3 groups, 15 timelines. Each audio DPT carries note events as a child
timeline, making them accessible for matching in Part II.

**Next:** Part II builds the Score group and aligns each recording via note matching.

***
# Part II: Score Group + Alignment to Recordings (Group 4)

The score group brings together three representations of the same music:

- **CLT1**: ABC v2.6 score (notes, measures, harmonies) — `ContinuousLogicalTimeline`
- **DGT1**: OMR ground truth (3,190 note heads across 22 pages) — `DiscreteGraphicalTimeline`
- **OpenScore**: OpenScore String Quartet edition (4th movement) — `ContinuousLogicalTimeline`

All three go into one `TimelineGroup`. Cross-domain coordinate transfer
(pixels ↔ quarters ↔ seconds) works automatically via linear interpolation.

## 6. CLT1: ABC v2.6 Score

In [7]:
ABC_DIR = DATA_DIR / "ABC"
abc_loader = Ms3Loader.from_file(
    ABC_DIR / "n04op18-4_04.notes.tsv",
    ABC_DIR / "n04op18-4_04.measures.tsv",
    ABC_DIR / "n04op18-4_04.harmonies.tsv",
)
clt1 = abc_loader.create_timeline(uid="clt_1")
clt1

ContinuousLogicalTimeline(id='clt_1', length=1757/2, unit=quarters, events=0, children=2, cmaps=1)

### 6.1 ABC Flow Control: Repeat Structure

The ABC score has repeats and volta brackets. The loader writes them as
the timeline's flow control, and `get_flow()` plays it: the default flow
takes all repeats. It returns the stretches of the score in the order
played and what playing found. The same is done for CLT2 (the recordings
edition) in Part III.

In [8]:
abc_flow = clt1.get_flow()
{"flow": abc_flow, "findings": abc_flow.findings}

{'flow': PlayedFlow(timeline='clt_1', 11 chunks, length=1116 quarters, 0 findings),
 'findings': ()}

`create_flow_maps_from_flow_control()` writes the score's atomic sections,
the stretches its flow control never divides, as Regions named A to M, and
the flows as FlowMaps listing them. §9.1 and §14.4 locate the sections.

In [9]:
abc_sections = clt1.create_flow_maps_from_flow_control()
section_names = {section.id: section.name for section in abc_sections.sections}
"".join(section_names[chunk] for chunk in abc_sections.flow_maps[0].chunks)

'AABBCCDEDFGGHIHJKLKM'

## 7. DGT1: OMR Ground Truth

The OMR data contains 3,190 note head bounding boxes across 22 score pages.
Each page has 2 systems (except the last which has 1), giving 43 system
segments in reading order. Note events use `Left` (start) and `Width`
(duration) as pixel coordinates. Each system's `onset_beats` values
provide a c-map from pixels to quarters.

**Architecture:** `SegmentLine[SegmentLine[DiscreteGraphicalTimeline]]` →
22 page `SegmentLine[DiscreteGraphicalTimeline]` segments → 2 system sub-segments each.

In [10]:
OMR_CSV = DATA_DIR / "OMR_groundtruth" / "OMR_xml_by_score" / "omr_note_heads.csv"
OMR_IMAGES = DATA_DIR / "OMR_groundtruth" / "Images"
omr_df = pd.read_csv(OMR_CSV)
IMAGE_WIDTH = Image.open(next(OMR_IMAGES.glob("*.png"))).size[0]

Build the DGT1 bottom-up: system segments →
page `SegmentLine[DiscreteGraphicalTimeline]` →
top-level `SegmentLine[SegmentLine[DiscreteGraphicalTimeline]]`.
Events and c-maps must be added **before** a timeline is locked as a child.

In [11]:
noteheads = pd.DataFrame(
    {
        "start": omr_df["Nodes.Node.Left"].astype(int),
        "end": (omr_df["Nodes.Node.Left"] + omr_df["Nodes.Node.Width"]).astype(int),
        "onset_beats": omr_df["onset_beats"].astype(float),
        "pitch": omr_df["pitch"],
        "staff_id": omr_df["staff_id"].astype(int),
        "midi_pitch": omr_df["midi_pitch_code"].astype(int),
        "top": omr_df["Nodes.Node.Top"].astype(int),
        "page": omr_df["@pageIndex"],
        "spacing_run_id": omr_df["spacing_run_id"],
    }
)

dgt1 = SegmentLine[SegmentLine[DiscreteGraphicalTimeline]](
    length=0,
    unit=TimeUnit.pixels,
    number_type=NumberType.int,
    uid="dgt_1",
)

for page_idx, page_data in noteheads.groupby("page", sort=True):
    # Systems ordered by vertical position (top first = reading order)
    sys_top = page_data.groupby("spacing_run_id")["top"].min()
    sys_order = sys_top.sort_values().index

    page = SegmentLine[DiscreteGraphicalTimeline](
        length=0,
        unit=TimeUnit.pixels,
        number_type=NumberType.int,
    )

    for sys_rank, sys_id in enumerate(sys_order):
        sys_data = page_data[page_data["spacing_run_id"] == sys_id]

        system = DiscreteGraphicalTimeline(
            length=IMAGE_WIDTH,
            uid=f"p{page_idx}_s{sys_rank}",
            name=f"Page {page_idx + 1}, System {sys_rank + 1}",
        )

        events = sys_data.drop(columns=["page", "spacing_run_id"])
        system.add_events(events.assign(event_type="Notehead").to_dict("records"))

        # C-map: pixels → quarters (deduplicated for chords at the same x)
        pairs = (
            events[["start", "onset_beats"]]
            .drop_duplicates("start")
            .sort_values("start")
        )
        if len(pairs) >= 2:
            system.add_conversion_map(
                TableMap(
                    x_values=pairs["start"].tolist(),
                    y_values=pairs["onset_beats"].tolist(),
                    source_unit="pixels",
                    target_unit="quarters",
                    uid=f"p{page_idx}_s{sys_rank}_px_to_qb",
                )
            )

        page.append_segment(system)

    dgt1.append_segment(page, name=f"page_{page_idx}")

dgt1.diagram(depth=True)

SegmentLine[SegmentLine[DiscreteGraphicalTimeline]][dgt_1] (22 children)
                          0 ∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶∶ 106425 pixels
  ├─ page_0               0 ∶                            4950
  │  ├─ Page 1, S...      0 ∶                            2475 (66 events)
  │  └─ Page 1, S...   2475 ∶                            4950 (68 events)
  ├─ page_1            4950  ∶                           9900
  │  ├─ Page 2, S...   4950  ∶                           7425 (76 events)
  │  └─ Page 2, S...   7425  ∶                           9900 (60 events)
  ├─ page_2            9900   ∶                          14850
  │  ├─ Page 3, S...   9900   ∶                          12375 (57 events)
  │  └─ Page 3, S...  12375    ∶                         14850 (71 events)
  │  ... (16 more children)
  ├─ page_19          94050                         ∶∶   99000
  │  ├─ Page 20, ...  94050                         ∶    96525 (81 events)
  │  └─ Page 20, ...  96525                          ∶   99000 (64 events)
  ├─ page_20          99000                           ∶  103950
  │  ├─ Page 21, ...  99000                           ∶  101475 (80 events)
  │  └─ Page 21, ... 101475                           ∶  103950 (53 events)
  └─ page_21         103950                            ∶ 106425
     └─ Page 22, ... 103950                            ∶ 106425 (4 events)

## 8. OpenScore (4th Movement Only)

The OpenScore edition covers all 4 movements. Its flow control marks the
movement boundaries as section breaks; we extract the 4th movement as a
child timeline.

In [12]:
OPENSCORE_DIR = DATA_DIR / "OpenScoreSQ"
os_loader = Ms3Loader.from_file(
    OPENSCORE_DIR / "sq8913219.notes.tsv",
    OPENSCORE_DIR / "sq8913219.measures.tsv",
)
os_full = os_loader.create_timeline(uid="openscore_full")
os_full

ContinuousLogicalTimeline(id='openscore_full', length=2447, unit=quarters, events=11898, children=0)

The loader writes the score's section ends as section breaks of its flow
control. Splitting at those coordinates creates one region per movement,
named `movement_1` to `movement_4`; the child is taken from the fourth by
its id.

In [13]:
boundaries = [brk.coordinate.value for brk in os_full.flow_control.section_breaks]
movements = os_full.create_regions_from_boundaries(
    [0, *[float(b) for b in boundaries], float(os_full.length.value)], prefix="movement"
)
openscore = os_full.create_child_from_region(movements[3].id, uid="openscore")
openscore

ContinuousLogicalTimeline(id='openscore', length=1757/2, unit=quarters, events=3156, children=0)

The four movement regions and the extracted child timeline:

In [14]:
os_full.diagram(show={"regions", "children"}, depth=1)

ContinuousLogicalTimeline[openscore_full] (15054 events (11898 own), 1 children, 4 regions)
                      0 ________________________________ 2447 quarters
  └─ movement_4   1568.5                     ____________ 2447 (3156 events)
  ┄ movement_1       0 ▐═════════▌                      880
  ┄ movement_2     880            ▐═══▌                 1271.5
  ┄ movement_3   1271.5                 ▐══▌             1568.5
  ┄ movement_4   1568.5                     ▐══════════▌ 2447

## 9. Score Group (Group 4)

All three score representations in one `TimelineGroup`. Cross-domain
coordinate transfer (pixels ↔ quarters) works via linear interpolation.

In [15]:
score_group = TimelineGroup(
    id="score",
    name="Score (ABC + OMR + OpenScore)",
    timelines=[clt1, dgt1, openscore],
)
score_group

TimelineGroup(id='score', n_timelines=3)

### 9.1 Cross-Domain Section Boundaries (Quarters → Pixels → Pages)

The atomic sections' starts (from §6.1) can now be mapped
through the score group to DGT1 pixel coordinates. This demonstrates
cross-domain coordinate transfer within a `TimelineGroup`: every member
spans the whole group, so a position on CLT1 (quarters) converts to DGT1
(pixels) linearly, by the ratio of the two lengths.

In [16]:
# Build a page-boundary lookup from DGT1's segment structure
_page_bounds = []
for _seg_id in dgt1.list_segments():
    _off = dgt1.get_child_offset(_seg_id)
    _seg = dgt1.get_child(_seg_id)
    _page_bounds.append(
        (float(_off.value), float(_off.value) + float(_seg.length.value))
    )

_section_rows = []
for _section in abc_sections.sections:
    _sid, _qb = _section.name, _section.start.value
    _ts = score_group.get_timestamp_at(_qb, "clt_1")
    _px = (
        _ts.get_coordinate_for("dgt_1", format="int")
        if "dgt_1" in _ts.present_timelines
        else None
    )
    _page = next(
        (
            i + 1
            for i, (s, e) in enumerate(_page_bounds)
            if _px is not None and s <= _px < e
        ),
        "-",
    )
    _section_rows.append(
        {"section": _sid, "quarters": float(_qb), "dgt1_pixels": _px, "page": _page}
    )
section_boundary_table = pd.DataFrame(_section_rows).set_index("section")
section_boundary_table

,quarters,dgt1_pixels,page
section,,,
A,0.0,0,1
B,32.0,3877,1
C,64.0,7753,2
D,96.0,11630,3
E,157.0,19020,4
F,160.0,19383,4
G,291.5,35313,8
H,315.5,38221,8
I,344.0,41674,9


Each atomic section's start coordinate is located precisely on a
specific page of the OMR score image. The pixel column gives the
linearised x-coordinate across all 22 pages; the page column tells
which score image to open.

### 9.2 The Score Played Through

The recordings play the score through, repeats and all, so their notes are
matched against the score's notes in the order played. A FlowMap states
that order over the folded score but holds no events: the played-through
score is a timeline of its own, `clt1_unfolded`, as long as the default
flow of §6.1. It forms the **played-through score group**; the folded score
group above, with the OMR pages and OpenScore, stays as it is.

The notes come from the edition's unfolded notes table, which counts its
`quarterbeats_playthrough` along the same axis.

In [17]:
clt1_unfolded = ContinuousLogicalTimeline(
    length=abc_flow.length, uid="clt1_unfolded", name="ABC v2.6, played through"
)
score_group_unfolded = TimelineGroup(
    id="score_unfolded",
    name="Score played through",
    timelines=[clt1_unfolded],
)
abc_notes_df = pd.read_csv(ABC_DIR / "n04op18-4_04_unfolded.notes.tsv", sep="\t")
abc_prepared = prepare_abc_notes_for_matching(abc_notes_df)
{
    "played-through length": clt1_unfolded.length,
    "sounding onsets": len(abc_prepared),  # untied notes and tie starts
}

{'played-through length': Coordinate(Fraction(1116, 1), quarters),
 'sounding onsets': 3751}

## 10. Aligning Recordings with the Score via Note Matching

Each EEP recording's note events (seconds, pitch, staff) are matched
against the ABC **unfolded** score notes (quarterbeats, pitch, staff)
prepared in §9.2 using greedy sequential matching. The result:
`MatchClaim` objects that connect recording positions to positions of the
played-through score.

Match each recording against the score. The `source_timeline` and
`target_timeline` are the audio DPT and the played-through CLT1 respectively —
the two timelines of every resulting `MatchClaim`.

A MatchClaim joins two events, but here neither side names its events
uniquely: the EEP note IDs repeat across instruments, and the unfolded
score repeats a passage's notes under the same IDs. Both sides are
therefore given as positions: each side is the sentinel CHUNK, whose
coordinate the claim's anchor states, and no event is added to either
timeline.
The EEP onsets are stated in seconds (`source_unit`); the audio DPT's
conversion map places them on its own axis.

We use `rv.store.notes` to access the EEP notes from the XML manifest's
score section — no direct `EepNotesLoader` import needed.

In [18]:
match_results = {}
for xml_path, group, dpt_id in [
    (NORMAL_XML, normal_group, "dpt_1"),
    (MECHANICAL_XML, mechanical_group, "dpt_6"),
    (EXAGGERATED_XML, exaggerated_group, "dpt_11"),
]:
    rv = RepoVizzLoader.from_file(xml_path)
    eep_events = rv.store.notes.to_dataframe()
    eep_prepared = prepare_eep_notes_for_matching(eep_events)
    match_results[dpt_id] = match_notes_by_attributes(
        eep_prepared,
        abc_prepared,
        match_columns=["pitch", "staff"],
        source_coord_column="start",
        target_coord_column="quarterbeats_playthrough",
        source_timeline=group.get_timeline(dpt_id),
        target_timeline=clt1_unfolded,
        source_unit=TimeUnit.seconds,
        target_unit=TimeUnit.quarters,
    )

normal_match = match_results["dpt_1"]
mechanical_match = match_results["dpt_6"]
exaggerated_match = match_results["dpt_11"]

In [19]:
{
    "Normal": normal_match.summary(),
    "Mechanical": mechanical_match.summary(),
    "Exaggerated": exaggerated_match.summary(),
}

{'Normal': {'matched': 3741,
  'unmatched_source': 15,
  'unmatched_target': 10,
  'match_claims': 3741},
 'Mechanical': {'matched': 3742,
  'unmatched_source': 14,
  'unmatched_target': 9,
  'match_claims': 3742},
 'Exaggerated': {'matched': 2650,
  'unmatched_source': 4,
  'unmatched_target': 1101,
  'match_claims': 2650}}

## Part II Summary

The score group unites 3 score representations across 2 domains (Logical +
Graphical). Note matching produced MatchClaims connecting each recording
group's audio timeline to the played-through CLT1 of §9.2:

| Recording | Matched | Unmatched EEP | Unmatched ABC |
|-----------|---------|---------------|---------------|
| Normal | 3,741 | 15 | 10 |
| Mechanical | 3,742 | 14 | 9 |
| Exaggerated | 2,650 | 4 | 1,101 |

**Next:** Part III adds the Emerson group and demonstrates cross-group
coordinate transfer using an `AlignmentBundle`.

***
# Part III: Emerson Recording + Cascading Alignment (Group 5)

The Emerson group connects a commercial recording to a second score
edition via segment-level alignment. Unlike the EEP groups (per-note
alignment), the Emerson recording is aligned at the level of 10
structural sections (alpha through kappa), derived from the score's
repeat structure.

The central payoff of this notebook is **cascading alignment**: by
adding the recordings edition's played-through score (CLT2) to the same
group as the played-through CLT1, coordinate transfer chains automatically
from the EEP recordings through both score editions to the Emerson
recording.

- **CLT2**: ABC v1.0 ("recordings edition") score — `ContinuousLogicalTimeline`
- **DPT16**: Emerson String Quartet recording (DG 1997) — `ContinuousPhysicalTimeline`

## 11. Building the Emerson Recording Components

### 11.1 CLT2: Recordings Edition Score

The recordings edition uses the same measure/repeat structure as CLT1 but
was encoded independently (ABC v1.0). We load it via Ms3Loader and play
its flow control.

In [20]:
REC_DIR = DATA_DIR / "recordings"
rec_loader = Ms3Loader.from_file(
    REC_DIR / "Beethoven_Op018No4-04.notes.tsv",
    REC_DIR / "Beethoven_Op018No4-04.measures.tsv",
    REC_DIR / "Beethoven_Op018No4-04.harmonies.tsv",
)
clt2 = rec_loader.create_timeline(uid="clt_2")
clt2

ContinuousLogicalTimeline(id='clt_2', length=876, unit=quarters, events=0, children=2, cmaps=1)

### 11.2 Flow Control: Inspect the Score's Repeat Structure

The loader writes the repeat signs and brackets of the measure data as the
timeline's flow control.

In [21]:
clt2.flow_control

FlowControlRegistry(9 jumps, 9 jump_targets, 2 breaks, 0 section_breaks, 1 barlines)

Play the default flow (all repeats taken) and a single-pass flow
(no repeats, last volta only) for comparison:

In [22]:
default_flow = clt2.get_flow()
single_flow = clt2.get_flow("single")
{"default": default_flow, "single": single_flow}

{'default': PlayedFlow(timeline='clt_2', 10 chunks, length=1116 quarters, 0 findings),
 'single': PlayedFlow(timeline='clt_2', 3 chunks, length=868 quarters, 2 findings)}

### 11.3 CLT2 Played Through

The recordings edition has the same repeat structure as CLT1. Its
played-through timeline is as long as its default flow (all repeats
taken), with coordinates in quarter-beats, suitable for the Emerson CSV's
unfolded section boundaries.

In [23]:
clt2_unfolded = ContinuousLogicalTimeline(
    length=default_flow.length,
    uid="clt2_unfolded",
    name="ABC v1.0, played through",
)
clt2_unfolded

ContinuousLogicalTimeline(id='clt2_unfolded', length=1116, unit=quarters, events=0, children=0)

### 11.4 DPT16: Emerson Recording

The `measureMapAudio.csv` provides a 10-segment alignment between the
unfolded score (floating measures) and the Emerson recording (seconds).
Each segment is labelled with a Greek letter (alpha through kappa).

In [24]:
ema_df = pd.read_csv(
    REC_DIR / "Beethoven_Op018No4-04_EmersonStringQuartet_DG_measureMapAudio.csv",
    sep="\t",
    index_col=0,
)
ema_df

,measure_score_start,measure_score_end,measure_unfold_start,measure_unfold_end,seconds_start,seconds_end
α,0.75,8.750,0.75,8.750,0.567007,7.381043
β,0.75,16.750,8.75,24.750,7.381043,21.823855
γ,8.75,24.750,24.75,40.750,21.823855,37.495283
δ,16.75,40.999,40.75,64.999,37.495283,59.309161
ε,25.00,39.999,65.00,79.999,59.309161,72.699388
ζ,41.00,79.750,80.00,118.750,72.699388,106.863129
η,73.75,87.750,118.75,132.750,106.863129,118.964393
θ,79.75,95.999,132.75,148.999,118.964393,132.918685
ι,88.00,94.999,149.00,155.999,132.918685,138.739229
κ,96.00,218.250,156.00,278.250,138.739229,241.823152


Create DPT16 as a `ContinuousPhysicalTimeline` in seconds. Unlike
the EEP recordings (per-note alignment), the Emerson alignment
operates at the level of section boundaries — the coordinates in
`ema_df` will become MatchClaims in §11.5 rather than a C-map.

In [25]:
dpt16_duration = float(ema_df["seconds_end"].iloc[-1])
dpt16 = ContinuousPhysicalTimeline(length=dpt16_duration, uid="dpt_16")
dpt16

ContinuousPhysicalTimeline(id='dpt_16', length=241.823151927, unit=seconds, events=0, children=0)

### 11.5 Emerson MatchClaims (alpha through kappa)

Each row in the measure-map CSV defines a section boundary: a
correspondence between an unfolded position on CLT2 and a seconds
position on DPT16. There is one MatchClaim per boundary, plus the final
end boundary.

These claims are the key connection between the Emerson recording and the
score group. Neither timeline has an event at a section boundary, so each
side is given as a position, the score side named by its section's Greek
letter. When the claims are made in §13, both sides of each claim are the
sentinel CHUNK: the anchor states the two coordinates, and no event is
added. The name stays on the claim's score side (`event_a_name`), naming
the position it states.

In [26]:
emerson_pairs = [
    (
        {"start": float(row["measure_unfold_start"]), "name": section},
        "clt2_unfolded",
        {"start": float(row["seconds_start"])},
        "dpt_16",
    )
    for section, row in ema_df.iterrows()
]

# Final end boundary
emerson_pairs.append(
    (
        {"start": float(ema_df["measure_unfold_end"].iloc[-1]), "name": "end"},
        "clt2_unfolded",
        {"start": float(ema_df["seconds_end"].iloc[-1])},
        "dpt_16",
    )
)

len(emerson_pairs)

11

## 12. Bridging the Two AlignmentBundles

### 12.1 The Key Move: Adding CLT2\_unfolded to the Played-Through Score Group

The played-through score group and the Emerson Group are currently independent:
neither shares a timeline with the other, and no MatchClaims connect them.
The Emerson MatchClaims (§11.5) link CLT2\_unfolded to DPT16 — but
CLT2\_unfolded is not yet in any group that the bundle's existing WarpMaps
can reach.

The insight: CLT1\_unfolded and CLT2\_unfolded encode the *same music*
from different editions. By adding CLT2\_unfolded to the played-through
score group, any coordinate on CLT1\_unfolded can be transferred to
CLT2\_unfolded via within-group interpolation, and from there to DPT16
via the Emerson MatchLine's WarpMap. The cascading path:

**DPT1 -> (WarpMap) -> CLT1\_unfolded -> (interpolation) -> CLT2\_unfolded -> (WarpMap) -> DPT16**

A single additional group membership retroactively enriches every
timeline in both groups.

In [27]:
score_group_unfolded.add_timeline(clt2_unfolded)
score_group_unfolded

TimelineGroup(id='score_unfolded', n_timelines=2)

CLT2\_unfolded now appears alongside CLT1\_unfolded in the played-through
score group. Every member spans the whole group, so the group links the two
editions' played-through timelines.

### 12.2 The Emerson Group

The Emerson group contains only DPT16 — the recording timeline.
CLT2\_unfolded lives in the score group, and the Emerson MatchClaims
connect the two groups.

In [28]:
emerson_group = TimelineGroup(
    id="emerson",
    name="Emerson Recording (DG 1997)",
    timelines=[dpt16],
)
emerson_group

TimelineGroup(id='emerson', n_timelines=1)

## 13. The AlignmentBundle

The bundle collects all 5 groups and connects them via MatchClaims.
Within each group, coordinate transfer uses linear interpolation.
Between groups, WarpMaps (built from MatchClaims) enable cross-domain
transfer.

In [29]:
bundle = AlignmentBundle(name="Beethoven Op.18/4 — Multimodal Alignment")

bundle.add_group(score_group_unfolded)
bundle.add_group(normal_group)
bundle.add_group(mechanical_group)
bundle.add_group(exaggerated_group)
bundle.add_group(emerson_group)

# Add EEP recording <-> CLT1 match claims
for dpt_id in ["dpt_1", "dpt_6", "dpt_11"]:
    bundle.add_match_claims(match_results[dpt_id].match_claims)

# Make the Emerson section boundary claims (CLT2_unfolded <-> DPT16), each
# between two positions
emerson_claims = bundle.create_match_claims(
    emerson_pairs,
    provenance=Provenance(
        method="imported",
        note="measureMapAudio.csv section boundaries",
    ),
)

bundle

AlignmentBundle(id='bundle_0re5c1lcigj47', name='Beethoven Op.18/4 — Multimodal Alignment', timelines=18, groups=5)

Match claims per connection:

In [30]:
pd.DataFrame(
    [
        {
            "recording": name,
            "source": dpt_id,
            "target": "clt1_unfolded",
            "matched": match_results[dpt_id].n_matched,
            "unmatched_source": match_results[dpt_id].n_unmatched_source,
            "unmatched_target": match_results[dpt_id].n_unmatched_target,
        }
        for name, dpt_id in [
            ("Normal", "dpt_1"),
            ("Mechanical", "dpt_6"),
            ("Exaggerated", "dpt_11"),
        ]
    ]
    + [
        {
            "recording": "Emerson",
            "source": "clt2_unfolded",
            "target": "dpt_16",
            "matched": len(emerson_claims),
            "unmatched_source": 0,
            "unmatched_target": 0,
        }
    ]
).set_index("recording")

,source,target,matched,unmatched_source,unmatched_target
recording,,,,,
Normal,dpt_1,clt1_unfolded,3741,15,10
Mechanical,dpt_6,clt1_unfolded,3742,14,9
Exaggerated,dpt_11,clt1_unfolded,2650,4,1101
Emerson,clt2_unfolded,dpt_16,11,0,0


### 13.1 Explicit MatchLine and WarpMap

Before demonstrating bundle-level coordinate transfer, it is
instructive to see the intermediate MatchLine and WarpMap that the
bundle constructs internally. The MatchLine orders the 11 Emerson
anchors by source coordinate; the WarpMap interpolates between them.

In [31]:
emerson_matchline = MatchLine.from_claims(
    emerson_claims, source_timeline_id="clt2_unfolded"
)
emerson_matchline

MatchLine(source='clt2_unfolded', stamps=11, targets=[dpt_16])

In [32]:
emerson_warpmap = WarpMap.from_match_line(
    emerson_matchline, target_timeline_id="dpt_16"
)
emerson_warpmap

WarpMap(source='clt2_unfolded', target='dpt_16', n_anchors=11)

Verify the WarpMap manually: transfer a coordinate from
CLT2\_unfolded to DPT16 and compare with a known section boundary:

In [33]:
# The first section boundary from ema_df
first_fm = float(ema_df["measure_unfold_start"].iloc[0])
first_sec = float(ema_df["seconds_start"].iloc[0])
transferred = emerson_warpmap.get_coordinate_at(first_fm)
{
    "CLT2_unfolded (floating measures)": first_fm,
    "DPT16 expected (seconds)": first_sec,
    "DPT16 via WarpMap": transferred,
}

{'CLT2_unfolded (floating measures)': 0.75,
 'DPT16 expected (seconds)': 0.567006803,
 'DPT16 via WarpMap': IdCoordinate(0.567006803, seconds, 'dpt_16')}

## 14. Cross-Group Coordinate Transfer

The bundle's `get_matchstamp_at()` method is the primary interface for
cross-domain coordinate transfer. Given a coordinate on any timeline,
it returns a `MatchStamp` with corresponding coordinates on all connected
timelines —
regardless of domain. With CLT2\_unfolded bridging the score group
and the Emerson MatchClaims, the bundle now reaches all 5 groups.

### 14.1 Inspecting CLT1's Harmony Annotations

Before transferring coordinates, let us see what harmonic events live
on CLT1. The annotations child carries all harmony labels from the
ABC score:

In [34]:
# Loader children are named after their category; their ids are minted.
annotations_id = next(
    child_id
    for child_id in clt1.list_children()
    if clt1.get_child(child_id).name == "annotations"
)
annotations_df = clt1.get_child(annotations_id).get_events().to_dataframe()
annotations_df[["start", "name"]].head(15)

,start,name
0,0,c.i
1,9,V65
2,10,i
3,11,V
4,12,i
5,13,V
6,17,i
7,21,v.iv
8,24,viio7/V
9,25,V(64)


### 14.2 USE CASE A — Transfer a Harmony Across All Groups

The `V7` at quarterbeat 79 (m. 20) is a dominant seventh — one of the
most recognisable sonorities. Quarterbeat 79 is a position of the folded
score; the default flow of §6.1 gives where it is played. It lies in a
repeated passage and is played twice:

In [35]:
v7_played = abc_flow.unfold(79)
v7_played

[Coordinate(Fraction(143, 1), quarters),
 Coordinate(Fraction(175, 1), quarters)]

Where does the first of these moments land across all 5 groups, in every
domain? The nested format groups results by `TimelineGroup`:

In [36]:
v7 = v7_played[0].value
stamp = bundle.get_matchstamp_at(v7, "clt1_unfolded")
len(stamp.present_timelines)

102

The bundle answers for every nested timeline of its trees, here also the
children of each recording, so the stamp holds 102 timelines. To keep the output readable, the cells below
narrow it to the bundle's own members (`bundle.timeline_ids`) with the
getter's `within=` filter, which always keeps the queried timeline and
selects among the others; the children's coordinates are still there
whenever you ask without the filter.

In [37]:
members = set(bundle.timeline_ids)
stamp = bundle.get_matchstamp_at(v7, "clt1_unfolded", within=members)
stamp

ID,Coordinate,Type
clt1_unfolded,143 quarters,stated
clt2_unfolded,143 quarters,inferred
dpt_1,1540191 samples,stated
dpt_2,1467 samples,inferred
dpt_3,2934 samples,inferred
dpt_4,6007 samples,inferred
dpt_5,8382 samples,inferred
dpt_6,1678084 samples,stated
dpt_7,1598 samples,inferred
dpt_8,3196 samples,inferred


`MatchStamp` belongs to the same unified stamp family as `TimeStamp`:
`get_coordinate()` returns a unit-bearing coordinate,
while `is_interpolated` reports whether resolution used interpolation.
Those two are independent. This position was reached by interpolation and is
still an exact `Fraction`, because the number type follows the quarters axis
and never records how a value was obtained. If you want to know whether a
reading was estimated, `is_interpolated` is the only thing that tells you.

In [38]:
{
    "CLT1_unfolded coordinate": stamp.get_coordinate("clt1_unfolded"),
    "interpolated": stamp.is_interpolated,
}

{'CLT1_unfolded coordinate': IdCoordinate(Fraction(143, 1), quarters, 'clt1_unfolded'),
 'interpolated': False}

`to_dict()` is serialization rather than a shortcut to numbers: every leaf is
the dictionary `{value, numerator, denominator, unit, number_type}`. The
float `value` is a mirror for consumers that only speak JSON numbers, the
ratio members carry the exact value where the axis is fraction-canonical, and
`number_type` names which side is authoritative. When you want a plain number
instead, ask for one with `get_coordinate_for(id, format="float")`.

In [39]:
bundle.get_matchstamp_at(v7, "clt1_unfolded", within=members).to_dict(format="nested")

{'score_unfolded': {'clt1_unfolded (quarters)': {'unit': 'quarters',
   'number_type': 'fraction',
   'value': [143, 1]},
  'clt2_unfolded (quarters)': {'unit': 'quarters',
   'number_type': 'fraction',
   'value': [143, 1]}},
 'normal': {'dpt_1 (samples)': {'unit': 'samples',
   'number_type': 'int',
   'value': 1540191},
  'dpt_2 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 1467},
  'dpt_3 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 2934},
  'dpt_4 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 6007},
  'dpt_5 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 8382}},
 'mechanical': {'dpt_6 (samples)': {'unit': 'samples',
   'number_type': 'int',
   'value': 1678084},
  'dpt_7 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 1598},
  'dpt_8 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 3196},
  'dpt_9 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 6545},
  'dpt_10 (samples)

Note that the `emerson` group now appears in the output: the
cascading path CLT1\_unfolded -> CLT2\_unfolded -> DPT16 connects the Emerson
recording to the rest of the bundle.

The flat format is useful for programmatic access:

In [40]:
bundle.get_matchstamp_at(v7, "clt1_unfolded", within=members).to_dict(format="flat")

{'clt1_unfolded (quarters)': {'unit': 'quarters',
  'number_type': 'fraction',
  'value': [143, 1]},
 'clt2_unfolded (quarters)': {'unit': 'quarters',
  'number_type': 'fraction',
  'value': [143, 1]},
 'dpt_1 (samples)': {'unit': 'samples',
  'number_type': 'int',
  'value': 1540191},
 'dpt_2 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 1467},
 'dpt_3 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 2934},
 'dpt_4 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 6007},
 'dpt_5 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 8382},
 'dpt_6 (samples)': {'unit': 'samples',
  'number_type': 'int',
  'value': 1678084},
 'dpt_7 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 1598},
 'dpt_8 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 3196},
 'dpt_9 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 6545},
 'dpt_10 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 9132},
 'd

### 14.3 USE CASE B — Reverse Transfer: Emerson to All Groups

The cascading alignment is bidirectional. Starting from a seconds
coordinate on DPT16 (the Emerson recording), we can reach every
connected timeline — including the three EEP recording groups:

In [41]:
bundle.get_matchstamp_at(120.0, "dpt_16", within=members).to_dict(format="nested")

{'emerson': {'dpt_16 (seconds)': {'unit': 'seconds',
   'number_type': 'float',
   'value': 120.0}},
 'score_unfolded': {'clt1_unfolded (quarters)': {'unit': 'quarters',
   'number_type': 'fraction',
   'value': [4713157070423973, 35184372088832]},
  'clt2_unfolded (quarters)': {'unit': 'quarters',
   'number_type': 'fraction',
   'value': [4713157070423973, 35184372088832]}},
 'normal': {'dpt_1 (samples)': {'unit': 'samples',
   'number_type': 'int',
   'value': 1450407},
  'dpt_2 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 1381},
  'dpt_3 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 2763},
  'dpt_4 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 5657},
  'dpt_5 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 7893}},
 'mechanical': {'dpt_6 (samples)': {'unit': 'samples',
   'number_type': 'int',
   'value': 1581315},
  'dpt_7 (samples)': {'unit': 'samples', 'number_type': 'int', 'value': 1506},
  'dpt_8 (samples)': {'u

A coordinate at 120 seconds into the Emerson recording is mapped
through the WarpMap to CLT2\_unfolded, then via interpolation to
CLT1\_unfolded, and from there via the per-note WarpMaps to DPT1, DPT6, and
DPT11 — all in a single call.

### 14.4 USE CASE C — Section Boundaries Across All Groups

The score's repeat structure defines atomic sections (A through M).
Resolved against CLT1, the default FlowMap of §6.1 places every section
where it is played; the first time each section is played gives its
**unfolded** quarterbeat start coordinate, a position of CLT1\_unfolded.
With the Emerson group now connected, the boundary table includes DPT16:

In [42]:
default_map = abc_sections.flow_maps[0]
section_coords = {}
for _chunk_id, _placed in zip(default_map.chunks, default_map.resolve(clt1).chunks):
    section_coords.setdefault(section_names[_chunk_id], _placed.start.value)
section_coords

{'A': Fraction(0, 1),
 'B': Fraction(64, 1),
 'C': Fraction(128, 1),
 'D': Fraction(192, 1),
 'E': Fraction(253, 1),
 'F': Fraction(317, 1),
 'G': Fraction(897, 2),
 'H': Fraction(993, 2),
 'I': Fraction(525, 1),
 'J': Fraction(557, 1),
 'K': Fraction(561, 1),
 'L': Fraction(589, 1),
 'M': Fraction(621, 1)}

In [43]:
boundary_rows = []
for _qb in section_coords.values():
    _stamp = bundle.get_matchstamp_at(_qb, "clt1_unfolded", within=members)
    _row = {}
    for _tl_id in _stamp.present_timelines:
        _coord = _stamp.get_coordinate_for(_tl_id)
        _row[f"{_tl_id} ({_coord.unit})"] = _coord.value
    boundary_rows.append(_row)
boundary_df = pd.DataFrame(
    boundary_rows,
    index=list(section_coords.keys()),
)
boundary_df.index.name = "section"
boundary_df

,clt1_unfolded (quarters),clt2_unfolded (quarters),dpt_1 (samples),dpt_2 (samples),dpt_3 (samples),dpt_4 (samples),dpt_5 (samples),dpt_6 (samples),dpt_7 (samples),dpt_8 (samples),dpt_9 (samples),dpt_10 (samples),dpt_11 (samples),dpt_12 (samples),dpt_13 (samples),dpt_14 (samples),dpt_15 (samples),dpt_16 (seconds)
section,,,,,,,,,,,,,,,,,,
A,0,0,44100,42,84,172,240,44100,42,84,172,240,44100,42,84,172,240,NaN
B,64,64,659662,628,1257,2573,3590,725796,691,1383,2831,3950,677580,645,1291,2643,3688,58.409620
C,128,128,1388046,1322,2644,5414,7554,1509834,1438,2876,5889,8217,1406333,1340,2679,5485,7654,114.858607
D,192,192,2055059,1957,3915,8016,11184,2232624,2126,4253,8708,12150,2063790,1966,3931,8050,11232,169.095231
E,253,253,2695244,2567,5134,10513,14668,2915972,2777,5554,11373,15869,2684332,2557,5113,10470,14609,220.531790
F,317,317,3368901,3209,6417,13140,18334,3625348,3453,6906,14140,19730,3334448,3176,6352,13006,18147,NaN
G,897/2,897/2,4998366,4761,9521,19496,27202,5393668,5137,10274,21037,29353,4963792,4728,9456,19361,27014,NaN
H,993/2,993/2,5252309,5003,10005,20486,28584,5669764,5400,10800,22114,30856,5208528,4962,9922,20316,28346,NaN
I,525,525,5529036,5266,10532,21566,30090,5566614,5302,10604,21712,30294,5117077,4874,9748,19959,27848,NaN


Each row gives the exact coordinate of a section boundary in every
member timeline and domain — including the Emerson recording's `dpt_16`
column. Every column matches its own timeline's native type, and that is
the point worth pausing on: sample counts are integers because those axes are
discrete, seconds are floats, and quarterbeats are exact fractions. One
cross-section can therefore hold three different number types at once without
any of them being a compromise. Nothing here was converted for display — each
value is the canonical one its axis declares.

## 15. Summary & Key Takeaways

> *"Any two events in the bundle can be related with each other —
> regardless of whether they live on the same timeline, in the same
> group, or even in the same domain — as long as a path of MatchClaims
> or ConversionMaps connects them."*

### The Cascading Alignment Pattern

The central demonstration of this notebook is that **a single additional
group membership retroactively enriches every timeline already present
in the bundle.** Adding CLT2\_unfolded to the played-through score group
bridges two independent alignment networks:

- **EEP recordings** (per-note MatchClaims) connect DPT1-DPT15 to CLT1\_unfolded
- **Emerson recording** (section-boundary MatchClaims) connects DPT16 to
  CLT2\_unfolded
- **CLT2\_unfolded in the score group** bridges the two via within-group
  interpolation

### Patterns Demonstrated

| Pattern | Example | Section |
|---------|---------|---------|
| `build_recording_group()` | Reusable factory for EEP recordings | 2-4 |
| `Ms3Loader.from_file()` | Load ABC score with notes, measures, annotations | 6 |
| `get_flow()` | The default flow of the score's flow control | 6.1 |
| `create_flow_maps_from_flow_control()` | Atomic sections as Regions, flows as FlowMaps | 6.1, 14.4 |
| `SegmentLine` nesting | OMR pages -> systems -> noteheads | 7 |
| Region extraction | OpenScore 4-movement -> movement 4 child | 8 |
| Cross-domain timestamps | Quarters -> pixels -> page number | 9.1 |
| Played-through score | A timeline as long as the default flow | 9.2, 11.3 |
| `match_notes_by_attributes()` | EEP <-> ABC note matching (played-through notes) | 10 |
| `create_match_claims()` with positions | Section-boundary alignment between positions (alpha-kappa) | 11.5, 13 |
| `add_timeline()` on a group | Bridge independent alignment networks | 12.1 |
| `MatchLine` + `WarpMap` | Explicit construction from MatchClaims | 13.1 |
| `AlignmentBundle` | Multi-group cross-domain transfer | 13 |
| `get_matchstamp_at()` | Universal coordinate resolution | 14 |
| Reverse transfer | DPT16 -> all groups | 14.3 |
| Cascading alignment | EEP <-> Score <-> Emerson via shared group | 12-14 |

**5 groups and 18 timelines in one bundle; with the folded score group,
3 domains.**